<a href="https://colab.research.google.com/github/clifcovickh/BiLSTMvsBiLSTMSentiment/blob/main/4_Dataset_Recovery_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q yfinance transformers deep-translator feedparser pandas numpy torch

import pandas as pd
import numpy as np
import feedparser
import yfinance as yf
import torch
import urllib.parse
import warnings
warnings.filterwarnings('ignore')
from transformers import pipeline
from deep_translator import GoogleTranslator


finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=0 if torch.cuda.is_available() else -1)
translator = GoogleTranslator(source='id', target='en')

master_df = pd.read_csv('BBCA_Master_Dataset_BiLSTM.csv')
master_df['Date'] = pd.to_datetime(master_df['Date'], format='mixed', dayfirst=True).dt.strftime('%Y-%m-%d')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.8 MB/s eta 0:00:00


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
search_keyword = "saham BBCA"
search_range = "14d"

print(f"Initiating Time Machine: Scraping Google News for the past {search_range}...")
encoded_search_keyword = urllib.parse.quote_plus(search_keyword)
rss_url = f"https://news.google.com/rss/search?q={encoded_search_keyword}+when:{search_range}&hl=id&gl=ID&ceid=ID:id"
feed = feedparser.parse(rss_url)

recovery_news = []

for entry in feed.entries:
    # Parse the publication date to YYYY-MM-DD
    pub_date = pd.to_datetime(entry.published).strftime('%Y-%m-%d')
    headline = entry.title

    try:
        english_text = translator.translate(headline)
        label = finbert(english_text)[0]['label']

        if label == 'positive': score = 1
        elif label == 'negative': score = -1
        else: score = 0
    except:
        score = 0

    recovery_news.append({'Date': pub_date, 'Sentiment_Score': score})

if not recovery_news:
    print(" No news found in the last 14 days.")
    df_recovery_sentiment = pd.DataFrame(columns=['Date', 'Sentiment_Score'])
else:
    # Convert to DataFrame and calculate the daily average for the missing days
    df_recovery_news = pd.DataFrame(recovery_news)
    df_recovery_sentiment = df_recovery_news.groupby('Date')['Sentiment_Score'].mean().reset_index()
    print(f" Successfully recovered and scored news for {len(df_recovery_sentiment)} different days.")
    display(df_recovery_sentiment.head())

Initiating Time Machine: Scraping Google News for the past 14d...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


 Successfully recovered and scored news for 14 different days.


,Date,Sentiment_Score
0,2026-02-24,-0.200000
1,2026-02-25,0.166667
2,2026-02-26,-0.333333
3,2026-02-27,-0.600000
4,2026-02-28,-1.000000


In [ ]:
bbca_data = yf.download("BBCA.JK", period="20d", progress=False)
if isinstance(bbca_data.columns, pd.MultiIndex):
    bbca_data.columns = bbca_data.columns.droplevel(1)

# Reset index so 'Date' becomes a normal column
bbca_data = bbca_data.reset_index()
bbca_data['Date'] = bbca_data['Date'].dt.strftime('%Y-%m-%d')

# Filter exactly the columns we need
recent_stocks = bbca_data[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()

print(f"Downloaded {len(recent_stocks)} recent trading days.")

Downloaded 20 recent trading days.


In [ ]:
print("Fusing recovered data...")

# Left join: Attach the recovered daily sentiment to the trading days
recovery_block = pd.merge(recent_stocks, df_recovery_sentiment, on='Date', how='left')
recovery_block['Sentiment_Score'] = recovery_block['Sentiment_Score'].fillna(0)

print("Patching the Master Database...")

# 1. Remove any rows in the Master CSV that have dates overlapping with our new recovery block
master_df = master_df[~master_df['Date'].isin(recovery_block['Date'])]

# 2. Append the fresh recovery block to the bottom
master_df = pd.concat([master_df, recovery_block], ignore_index=True)

# 3. Sort chronologically just to be absolutely safe
master_df = master_df.sort_values('Date').reset_index(drop=True)

# Save the healed database
master_df.to_csv('BBCA_Master_Dataset_LSTM.csv', index=False)

print("="*50)
print("The AI's short-term memory has been successfully restored.")
print("You may now run Notebook 3 (Live Analyst) to get tomorrow's prediction.")
print("="*50)
display(master_df.tail(5))

Fusing recovered data...
Patching the Master Database...
The AI's short-term memory has been successfully restored.
You may now run Notebook 3 (Live Analyst) to get tomorrow's prediction.


,Date,Open,High,Low,Close,Volume,Sentiment_Score
270,2026-03-04,7075.0,7075.0,6825.0,6875.0,216518700.0,-0.555556
271,2026-03-05,6975.0,7125.0,6950.0,7100.0,110256300.0,0.222222
272,2026-03-06,7075.0,7100.0,7000.0,7000.0,102725700.0,-0.222222
273,2026-03-09,6900.0,6950.0,6825.0,6875.0,150426400.0,0.083333
274,2026-03-10,6950.0,7050.0,6925.0,6950.0,68470800.0,0.000000
